In [2]:
pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 1.0/114.6 MB 7.5 MB/s eta 0:00:16
   - -------------------------------------- 2.9/114.6 MB 8.6 MB/s eta 0:00:13
   - -------------------------------------- 5.0/114.6 MB 8.6 MB/s eta 0:00:13
   -- ------------------------------------- 7.6/114.6 MB 10.2 MB/s eta 0:00:11
   --- ------------------------------------ 10.2/114.6 MB 10.2 MB/s eta 0:00:11
   ---- ----------------------------------- 13.1/114.6 MB 10.9 MB/s eta 0:00:10
   ----- ---------------------------------- 15.2/114.6 MB 10.7 MB/s eta 0:00:10
   ------ --------------------------------- 18.6/114.6 MB 11.3 MB/s eta 0:00:09
   ------ --------------------------------- 19.1/114.6 MB 10.4 MB/s eta 0:00:10
   ------- -------------------------------- 20.7/114.6 MB 10.2 MB/s eta 0:00:10
   ------- -------------------------------- 22.3/114.6 MB 9.8 MB/s eta 0:00:10
   -------- ------------------------------- 25.2/114.6 MB

In [3]:
# ===================== IMPORTS =====================
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ===================== LOAD DATA =====================
df = pd.read_csv("StudentsPerformance.csv")

# ===================== CLEAN =====================
df.columns = df.columns.str.strip().str.lower()
print("Columns:", df.columns.tolist())

# ===================== ENCODE =====================
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# ===================== SELECT TARGET COLUMN =====================
math_col = df.columns[5]   # 🔁 if wrong, change index after seeing printed columns
print("Using:", math_col)

# ===================== CREATE CLASSIFICATION TARGET =====================
df['pass'] = df[math_col].apply(lambda x: 1 if x >= 40 else 0)

X = df.drop([math_col, 'pass'], axis=1).values
y = df['pass'].values

# ===================== SPLIT =====================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).view(-1,1)

# ===================== MODEL =====================
class Classifier(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.out(x))
        return x

model = Classifier(X_train.shape[1])

# ===================== TRAIN =====================
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    preds = model(X_train)
    loss = criterion(preds, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# ===================== TEST =====================
with torch.no_grad():
    preds = model(X_test)
    preds = (preds > 0.5).float()
    acc = (preds == y_test).float().mean()

print("Classification Accuracy:", acc.item())

Columns: ['school', 'sex', 'age', 'address', 'famsize', 'pstatus', 'medu', 'fedu', 'mjob', 'fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'dalc', 'walc', 'health', 'absences', 'g1', 'g2', 'g3']
Using: pstatus
Classification Accuracy: 1.0
